# Fine-tune (LoRA)

Fine-tunes **Qwen2-1.5B** on GSM8K using LoRA. Artifacts go to `output/fine_tune/` (gitignored).

| Output | Contents |
|--------|---------|
| Cell output | tqdm bar + `[train] step=… loss=… lr=… epoch=…` per log event |
| `training_run.log` | Timestamped log, appended each run |
| `trainer_log_history.json` | Per-step `loss`, `learning_rate`, `epoch` from HF Trainer |
| `training_summary.json` | Hyperparameters + final loss / step / epoch |
| `adapter_*/`, checkpoints | Saved by HF Trainer under the run folder |

**Single run:** `RUN_HYPERPARAMETER_SWEEP = False` — uses defaults from `hyperparameters.py`.  
**Sweep:** `True` — one run per entry in `PARAM_COMBINATIONS`, each in its own subfolder (`run_000_…`).

Run cells top to bottom.

### 1 — Imports and callback

Imports the stack and defines `NotebookProgressCallback` (prints `[train]` lines per log event). Run once per session.

In [5]:
import json
import logging
from datetime import datetime, timezone
from pathlib import Path

import torch
import transformers
from transformers import TrainerCallback

from qwen_math_flow.download_model import download_qwen_2_15b
from qwen_math_flow.load_dataset import format_gsm8k_as_deterministic_json_chat, load_and_tokenize_math
from qwen_math_flow.lora_finetune import create_lora_model, run_finetune


class NotebookProgressCallback(TrainerCallback):
    """Print one line per Trainer log so loss / lr / epoch show in the notebook while training."""

    def on_train_begin(self, args, state, control, **kwargs):
        print(
            "[train] started | "
            f"epochs={args.num_train_epochs} | log every {args.logging_steps} step(s) | "
            "progress bar + metrics lines below",
            flush=True,
        )

    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs:
            return
        parts = [f"step={state.global_step}"]
        if logs.get("loss") is not None:
            parts.append(f"loss={logs['loss']:.4f}")
        if logs.get("learning_rate") is not None:
            parts.append(f"lr={logs['learning_rate']:.2e}")
        if logs.get("epoch") is not None:
            parts.append(f"epoch={logs['epoch']:.4f}")
        print("[train] " + " | ".join(parts), flush=True)

    def on_train_end(self, args, state, control, **kwargs):
        print(f"[train] finished | global_step={state.global_step}", flush=True)

### 2 — Config

Set the output path, sweep toggle, and logging verbosity. Training defaults (LR, LoRA rank, batch size, etc.) come from `hyperparameters.py`; entries in `PARAM_COMBINATIONS` override only the keys they list.

Re-run this cell whenever you change sweep settings.

In [6]:
from qwen_math_flow.hyperparameters import (
    DATASET_NAME,
    DATASET_SPLIT,
    GRADIENT_ACCUMULATION_STEPS,
    LEARNING_RATE,
    LOAD_IN_4BIT,
    LORA_ALPHA,
    LORA_R,
    MAX_LENGTH,
    MAX_TRAIN_SAMPLES,
    MODEL_CACHE_DIR,
    NUM_EPOCHS,
    PER_DEVICE_TRAIN_BATCH_SIZE,
)

OUTPUT_BASE = "output/fine_tune"

# False: one run using hyperparameters.py only. True: one run per dict in PARAM_COMBINATIONS.
RUN_HYPERPARAMETER_SWEEP = True

# Sweeps: each dict can override training keys; missing keys use hyperparameters.py defaults.
# Pragmatic 5-run subset from the original 18-combination grid.
PARAM_COMBINATIONS = [
    {
        "learning_rate": 1e-5,
        "lora_r": 8,
        "lora_alpha": 16,
        "num_epochs": 3,
        "max_train_samples": None,
    },
    {
        "learning_rate": 3e-5,
        "lora_r": 16,
        "lora_alpha": 32,
        "num_epochs": 3,
        "max_train_samples": None,
    },
    {
        "learning_rate": 5e-5,
        "lora_r": 32,
        "lora_alpha": 64,
        "num_epochs": 3,
        "max_train_samples": None,
    },
    {
        "learning_rate": 7e-5,
        "lora_r": 16,
        "lora_alpha": 32,
        "num_epochs": 3,
        "max_train_samples": None,
    },
    {
        "learning_rate": 1e-4,
        "lora_r": 32,
        "lora_alpha": 64,
        "num_epochs": 3,
        "max_train_samples": None,
    },
]
# Current size: 5 combinations.


def merged_training_params(overrides: dict) -> dict:
    """Defaults from hyperparameters.py, overridden by one sweep row."""
    return {
        "learning_rate": overrides.get("learning_rate", LEARNING_RATE),
        "lora_r": overrides.get("lora_r", LORA_R),
        "lora_alpha": overrides.get("lora_alpha", LORA_ALPHA),
        "num_epochs": overrides.get("num_epochs", 3),
        "per_device_train_batch_size": overrides.get(
            "per_device_train_batch_size", PER_DEVICE_TRAIN_BATCH_SIZE
        ),
        "gradient_accumulation_steps": overrides.get(
            "gradient_accumulation_steps", GRADIENT_ACCUMULATION_STEPS
        ),
        "max_length": overrides.get("max_length", MAX_LENGTH),
        "max_train_samples": overrides.get("max_train_samples", MAX_TRAIN_SAMPLES),
    }


def combination_subdir(index: int, m: dict) -> str:
    """Unique folder name for one combination."""
    lr = m["learning_rate"]
    lr_s = f"{lr:.0e}".replace("e-0", "e-").replace("e+0", "e+")
    return f"run_{index:03d}_lr{lr_s}_r{m['lora_r']}_a{m['lora_alpha']}_ep{m['num_epochs']}"


SWEEP_RUNS = PARAM_COMBINATIONS if RUN_HYPERPARAMETER_SWEEP else [{}]
NUM_COMBINATIONS = len(SWEEP_RUNS)

# Resume mode: skip run folders that already contain completed summaries.
RESUME_INCOMPLETE_ONLY = True

# Trainer: log loss / lr every N steps (1 = every step — most verbose)
LOGGING_STEPS = 1
SHOW_NOTEBOOK_LOG_LINES = True
DISABLE_TQDM = False
TRANSFORMERS_DEBUG = False

Path(OUTPUT_BASE).mkdir(parents=True, exist_ok=True)
print(f"Output base: {Path(OUTPUT_BASE).resolve()}")
print(f"RUN_HYPERPARAMETER_SWEEP={RUN_HYPERPARAMETER_SWEEP}  →  {NUM_COMBINATIONS} training run(s)")
if RUN_HYPERPARAMETER_SWEEP:
    for i, o in enumerate(PARAM_COMBINATIONS):
        print(f"  [{i}] {o}  →  {combination_subdir(i, merged_training_params(o))}")
print(f"RESUME_INCOMPLETE_ONLY={RESUME_INCOMPLETE_ONLY}")
print(
    f"LOGGING_STEPS={LOGGING_STEPS}  SHOW_NOTEBOOK_LOG_LINES={SHOW_NOTEBOOK_LOG_LINES}  "
    f"DISABLE_TQDM={DISABLE_TQDM}  TRANSFORMERS_DEBUG={TRANSFORMERS_DEBUG}"
)

Output base: C:\Users\Admin\Desktop\smu\LLM-project\output\fine_tune
RUN_HYPERPARAMETER_SWEEP=True  →  5 training run(s)
  [0] {'learning_rate': 1e-05, 'lora_r': 8, 'lora_alpha': 16, 'num_epochs': 3, 'max_train_samples': None}  →  run_000_lr1e-5_r8_a16_ep3
  [1] {'learning_rate': 3e-05, 'lora_r': 16, 'lora_alpha': 32, 'num_epochs': 3, 'max_train_samples': None}  →  run_001_lr3e-5_r16_a32_ep3
  [2] {'learning_rate': 5e-05, 'lora_r': 32, 'lora_alpha': 64, 'num_epochs': 3, 'max_train_samples': None}  →  run_002_lr5e-5_r32_a64_ep3
  [3] {'learning_rate': 7e-05, 'lora_r': 16, 'lora_alpha': 32, 'num_epochs': 3, 'max_train_samples': None}  →  run_003_lr7e-5_r16_a32_ep3
  [4] {'learning_rate': 0.0001, 'lora_r': 32, 'lora_alpha': 64, 'num_epochs': 3, 'max_train_samples': None}  →  run_004_lr1e-4_r32_a64_ep3
RESUME_INCOMPLETE_ONLY=True
LOGGING_STEPS=1  SHOW_NOTEBOOK_LOG_LINES=True  DISABLE_TQDM=False  TRANSFORMERS_DEBUG=False


### 3 — Train

Loads the model and data, runs `run_finetune` for each combination, and writes `training_summary.json`, `trainer_log_history.json`, and `training_run.log` per run. In sweep mode, also writes `sweep_index.json` at the top level. If `RESUME_INCOMPLETE_ONLY=True`, completed run folders are skipped automatically.

Re-run config first if hyperparameters changed.

In [8]:
def _setup_run_logging(log_path: Path) -> None:
    logging.basicConfig(
        level=logging.DEBUG if TRANSFORMERS_DEBUG else logging.INFO,
        format="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
        handlers=[
            logging.StreamHandler(),
            logging.FileHandler(log_path, encoding="utf-8", mode="a"),
        ],
        force=True,
    )
    for name in ("transformers", "transformers.trainer", "qwen_math_flow.lora_finetune"):
        logging.getLogger(name).setLevel(logging.DEBUG if TRANSFORMERS_DEBUG else logging.INFO)
    logging.getLogger("datasets").setLevel(logging.WARNING)
    if TRANSFORMERS_DEBUG:
        transformers.logging.set_verbosity_debug()
    else:
        transformers.logging.set_verbosity_info()


all_summaries = []
skipped_runs = 0


def _is_completed_run(out_dir: Path) -> bool:
    summary_path = out_dir / "training_summary.json"
    hist_path = out_dir / "trainer_log_history.json"
    if not summary_path.exists() or not hist_path.exists():
        return False
    try:
        with open(summary_path, "r", encoding="utf-8") as f:
            s = json.load(f)
        with open(hist_path, "r", encoding="utf-8") as f:
            h = json.load(f)
    except Exception:
        return False
    if not isinstance(h, list) or len(h) == 0:
        return False
    last = h[-1] if isinstance(h[-1], dict) else {}
    # Completion marker from Trainer's final summary record.
    return "train_loss" in last and s.get("global_step") is not None


for run_index, overrides in enumerate(SWEEP_RUNS):
    merged = merged_training_params(overrides)
    out = (
        Path(OUTPUT_BASE) / combination_subdir(run_index, merged)
        if RUN_HYPERPARAMETER_SWEEP
        else Path(OUTPUT_BASE)
    )
    out.mkdir(parents=True, exist_ok=True)

    if RESUME_INCOMPLETE_ONLY and _is_completed_run(out):
        print(f"[skip] run {run_index + 1}/{NUM_COMBINATIONS} already completed: {out}", flush=True)
        skipped_runs += 1
        continue

    log_file = out / "training_run.log"

    print(
        f"\n=== Run {run_index + 1}/{NUM_COMBINATIONS}  {out.name}  merged={merged} ===\n",
        flush=True,
    )
    _setup_run_logging(log_file)
    logging.info("Logging to %s", log_file.resolve())

    model, tokenizer = download_qwen_2_15b(
        cache_dir=MODEL_CACHE_DIR,
        load_in_4bit=LOAD_IN_4BIT,
        device_map="auto" if LOAD_IN_4BIT else None,
    )
    tokenized_train = load_and_tokenize_math(
        tokenizer,
        name=DATASET_NAME,
        split=DATASET_SPLIT,
        max_samples=merged["max_train_samples"],
        max_length=merged["max_length"],
        message_formatter=format_gsm8k_as_deterministic_json_chat,
    )
    cap_msg = (
        "full train split"
        if merged["max_train_samples"] is None
        else f"up to {merged['max_train_samples']} samples"
    )
    logging.info(
        "GSM8K training: %s samples (%s), %s epoch(s).",
        len(tokenized_train),
        cap_msg,
        merged["num_epochs"],
    )

    peft_model = create_lora_model(
        model,
        r=merged["lora_r"],
        lora_alpha=merged["lora_alpha"],
        use_4bit_or_8bit=LOAD_IN_4BIT,
    )
    callbacks = []
    if SHOW_NOTEBOOK_LOG_LINES:
        callbacks.append(NotebookProgressCallback())

    metrics = run_finetune(
        peft_model,
        tokenizer,
        tokenized_train,
        output_dir=str(out),
        num_epochs=merged["num_epochs"],
        per_device_train_batch_size=merged["per_device_train_batch_size"],
        gradient_accumulation_steps=merged["gradient_accumulation_steps"],
        learning_rate=merged["learning_rate"],
        logging_steps=LOGGING_STEPS,
        logging_first_step=True,
        logging_strategy="steps",
        disable_tqdm=DISABLE_TQDM,
        callbacks=callbacks,
        save_log_history_json=True,
        bf16=False,
        fp16=True
    )

    summary = {
        "run_index": run_index,
        "run_hyperparameter_sweep": RUN_HYPERPARAMETER_SWEEP,
        "sweep_overrides": overrides,
        "merged_params": merged,
        "finished_at_utc": datetime.now(timezone.utc).isoformat(),
        "output_dir": str(out.resolve()),
        "log_files": {
            "training_run_log": str((out / "training_run.log").resolve()),
            "trainer_log_history_json": str((out / "trainer_log_history.json").resolve()),
        },
        "logging_steps": LOGGING_STEPS,
        "show_notebook_log_lines": SHOW_NOTEBOOK_LOG_LINES,
        "disable_tqdm": DISABLE_TQDM,
        "transformers_debug": TRANSFORMERS_DEBUG,
        "dataset": DATASET_NAME,
        "split": DATASET_SPLIT,
        "num_train_samples": len(tokenized_train),
        "num_epochs": merged["num_epochs"],
        "per_device_train_batch_size": merged["per_device_train_batch_size"],
        "gradient_accumulation_steps": merged["gradient_accumulation_steps"],
        "learning_rate": merged["learning_rate"],
        "max_length": merged["max_length"],
        "lora_r": merged["lora_r"],
        "lora_alpha": merged["lora_alpha"],
        "load_in_4bit": LOAD_IN_4BIT,
        "final_train_loss": metrics.get("train_loss"),
        "global_step": metrics.get("global_step"),
        "epoch": metrics.get("epoch"),
    }
    with open(out / "training_summary.json", "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)
    all_summaries.append(summary)

    logging.info("Done run %s/%s: %s", run_index + 1, NUM_COMBINATIONS, out.resolve())
    del model, peft_model, tokenizer, tokenized_train
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

if RUN_HYPERPARAMETER_SWEEP:
    sweep_index_path = Path(OUTPUT_BASE) / "sweep_index.json"
    with open(sweep_index_path, "w", encoding="utf-8") as f:
        json.dump(
            {
                "finished_at_utc": datetime.now(timezone.utc).isoformat(),
                "num_runs": NUM_COMBINATIONS,
                "num_skipped": skipped_runs,
                "num_executed": len(all_summaries),
                "runs": [
                    {
                        "output_dir": s["output_dir"],
                        "final_train_loss": s.get("final_train_loss"),
                        "merged_params": s["merged_params"],
                    }
                    for s in all_summaries
                ],
            },
            f,
            indent=2,
        )
    print(f"Wrote sweep index: {sweep_index_path.resolve()}")

print(
    f"Finished: total={NUM_COMBINATIONS}, executed={len(all_summaries)}, skipped={skipped_runs}.\n"
    f"Artifacts under:\n  {Path(OUTPUT_BASE).resolve()}\n"
)


=== Run 1/5  run_000_lr1e-5_r8_a16_ep3  merged={'learning_rate': 1e-05, 'lora_r': 8, 'lora_alpha': 16, 'num_epochs': 3, 'per_device_train_batch_size': 4, 'gradient_accumulation_steps': 4, 'max_length': 512, 'max_train_samples': None} ===



2026-04-06 00:48:39 | INFO     | root | Logging to C:\Users\Admin\Desktop\smu\LLM-project\output\fine_tune\run_000_lr1e-5_r8_a16_ep3\training_run.log
2026-04-06 00:48:39 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2-1.5B/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-06 00:48:39 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2-1.5B/8a16abf2848eda07cc5253dec660bf1ce007ad7a/config.json "HTTP/1.1 200 OK"
loading configuration file config.json from cache at C:\Users\Admin\.cache\huggingface\hub\models--Qwen--Qwen2-1.5B\snapshots\8a16abf2848eda07cc5253dec660bf1ce007ad7a\config.json
Model config Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151643,
  "hidden_act": "silu",
  "hidden_size": 1536,
  "initializer_range": 0.02,
  "intermediate_size": 8960,
  "layer_types": [
    "full_atten

[train] started | epochs=3 | log every 1 step(s) | progress bar + metrics lines below


Step,Training Loss
1,1.683321
2,1.730360
3,1.738154
4,1.718107
5,1.686252
6,1.707746
7,1.637500
8,1.610989
9,1.702527
10,1.707852


[train] step=1 | loss=1.6833 | lr=0.00e+00 | epoch=0.0021
[train] step=2 | loss=1.7304 | lr=7.14e-08 | epoch=0.0043
[train] step=3 | loss=1.7382 | lr=1.43e-07 | epoch=0.0064
[train] step=4 | loss=1.7181 | lr=2.14e-07 | epoch=0.0086
[train] step=5 | loss=1.6863 | lr=2.86e-07 | epoch=0.0107
[train] step=6 | loss=1.7077 | lr=3.57e-07 | epoch=0.0128
[train] step=7 | loss=1.6375 | lr=4.29e-07 | epoch=0.0150
[train] step=8 | loss=1.6110 | lr=5.00e-07 | epoch=0.0171
[train] step=9 | loss=1.7025 | lr=5.71e-07 | epoch=0.0193
[train] step=10 | loss=1.7079 | lr=6.43e-07 | epoch=0.0214
[train] step=11 | loss=1.7555 | lr=7.14e-07 | epoch=0.0235
[train] step=12 | loss=1.8420 | lr=7.86e-07 | epoch=0.0257
[train] step=13 | loss=1.7097 | lr=8.57e-07 | epoch=0.0278
[train] step=14 | loss=1.7561 | lr=9.29e-07 | epoch=0.0300
[train] step=15 | loss=1.8109 | lr=1.00e-06 | epoch=0.0321
[train] step=16 | loss=1.7465 | lr=1.07e-06 | epoch=0.0342
[train] step=17 | loss=1.7255 | lr=1.14e-06 | epoch=0.0364
[train

...

[train] step=1399 | loss=0.0994 | lr=4.75e-07 | epoch=2.9909
[train] step=1400 | loss=0.1078 | lr=3.96e-07 | epoch=2.9930
[train] step=1401 | loss=0.0799 | lr=3.16e-07 | epoch=2.9952
[train] step=1402 | loss=0.0922 | lr=2.37e-07 | epoch=2.9973
[train] step=1403 | loss=0.1230 | lr=1.58e-07 | epoch=2.9995
[train] step=1404 | loss=0.0859 | lr=7.91e-08 | epoch=3.0000


Saving model checkpoint to output\fine_tune\run_004_lr1e-4_r32_a64_ep3\checkpoint-1404
2026-04-06 05:43:07 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2-1.5B/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-06 05:43:07 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2-1.5B/8a16abf2848eda07cc5253dec660bf1ce007ad7a/config.json "HTTP/1.1 200 OK"
2026-04-06 05:43:07 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2-1.5B/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-06 05:43:07 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2-1.5B/8a16abf2848eda07cc5253dec660bf1ce007ad7a/config.json "HTTP/1.1 200 OK"
loading configuration file config.json from cache at C:\Users\Admin\.cache\huggingface\hub\models--Qwen--Qwen2-1.5B\snapshots\8a16abf2848eda07cc5253dec660bf1ce007ad7a\config.json
Model config Qwen2Config {
 

[train] step=1404 | epoch=3.0000
[train] finished | global_step=1404


Saving model checkpoint to output\fine_tune\run_004_lr1e-4_r32_a64_ep3
2026-04-06 05:43:07 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2-1.5B/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-06 05:43:07 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2-1.5B/8a16abf2848eda07cc5253dec660bf1ce007ad7a/config.json "HTTP/1.1 200 OK"
2026-04-06 05:43:08 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2-1.5B/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-06 05:43:08 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2-1.5B/8a16abf2848eda07cc5253dec660bf1ce007ad7a/config.json "HTTP/1.1 200 OK"
loading configuration file config.json from cache at C:\Users\Admin\.cache\huggingface\hub\models--Qwen--Qwen2-1.5B\snapshots\8a16abf2848eda07cc5253dec660bf1ce007ad7a\config.json
Model config Qwen2Config {
  "architectures"

Wrote sweep index: C:\Users\Admin\Desktop\smu\LLM-project\output\fine_tune\sweep_index.json
Finished: total=5, executed=5, skipped=0.
Artifacts under:
  C:\Users\Admin\Desktop\smu\LLM-project\output\fine_tune

